# Experimental: local Gemma 3 270M method APIs

Reusable Python APIs from `local_llm` (in-process). Run cells top-to-bottom once so the model stays loaded.

| API | Purpose |
|---|---|
| `ensure_model` | download GGUF into `local_llm/models/` |
| `default_model_path` / `MODELS_DIR` / `MODEL_ID` | paths + model id |
| `detect_runtime` | Mac Metal / Linux CUDA-or-CPU knobs |
| `LocalGemma` | low-level engine: `.chat`, `.complete` |
| `LocalLLMClient` | preferred import for other modules |

In [88]:
from pathlib import Path
import sys

# Repo root on sys.path when the notebook kernel cwd is local_llm/ or repo root.
ROOT = Path.cwd().resolve()
if (ROOT / "local_llm").is_dir():
    pass
elif (ROOT.parent / "local_llm").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from local_llm import (
    LocalGemma,
    LocalLLMClient,
    MODEL_ID,
    MODELS_DIR,
    default_model_path,
    ensure_model,
)
from local_llm.engine import detect_runtime

print("MODEL_ID", MODEL_ID)
print("MODELS_DIR", MODELS_DIR)
print("default_model_path", default_model_path())

MODEL_ID gemma3:270m
MODELS_DIR /Users/shravanchaudhary/Documents/work/galadriel-public/local_llm/models
default_model_path /Users/shravanchaudhary/Documents/work/galadriel-public/local_llm/models/google_gemma-3-270m-it-qat-Q4_K_M.gguf


## `ensure_model`
Downloads the QAT GGUF into the repo folder if missing. Safe to re-run.

In [89]:
model_path = ensure_model()
print(model_path)
print(f"{model_path.stat().st_size / (1024 * 1024):.1f} MB")

/Users/shravanchaudhary/Documents/work/galadriel-public/local_llm/models/google_gemma-3-270m-it-qat-Q4_K_M.gguf
241.4 MB


## `detect_runtime`
Platform tuning used when loading llama.cpp (Metal / CUDA / CPU).

In [90]:
detect_runtime()

{'system': 'darwin',
 'machine': 'arm64',
 'backend': 'metal',
 'n_threads': 9,
 'n_gpu_layers': -1,
 'flash_attn': True,
 'use_mmap': True,
 'use_mlock': False}

## `LocalGemma` — load once
Reuse this `llm` in the cells below.

In [91]:
llm = LocalGemma()  # ensure=True downloads if needed
llm.model_id, llm.model_path, llm.runtime

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


('gemma3:270m',
 PosixPath('/Users/shravanchaudhary/Documents/work/galadriel-public/local_llm/models/google_gemma-3-270m-it-qat-Q4_K_M.gguf'),
 {'system': 'darwin',
  'machine': 'arm64',
  'backend': 'metal',
  'n_threads': 9,
  'n_gpu_layers': -1,
  'flash_attn': True,
  'use_mmap': True,
  'use_mlock': False})

## `LocalGemma.complete`
Raw text completion (no chat template).

In [92]:
result = llm.complete(
    "Write one short sentence about staging deploys:\n",
    max_tokens=64,
    temperature=0.7,
)
print(result.text)
print(
    {
        "finish_reason": result.finish_reason,
        "prompt_tokens": result.prompt_tokens,
        "completion_tokens": result.completion_tokens,
        "total_tokens": result.total_tokens,
    }
)

**Staging deploys are a critical aspect of the deployment process, ensuring the deployment is properly organized and executed. This includes ensuring the deployment plan is clearly defined, the deployment team is properly trained and supported, and the deployment is monitored and managed.**
The primary goal of staging deploys is to ensure the deployment plan
{'finish_reason': 'length', 'prompt_tokens': 11, 'completion_tokens': 64, 'total_tokens': 75}


## `LocalGemma.chat`
Chat messages (system / user / assistant). This is the usual call shape for other modules.

In [93]:
result = llm.chat(
    [
        {"role": "system", "content": "Answer in one short sentence."},
        {"role": "user", "content": "what can you do?"},
    ],
    max_tokens=96,
    temperature=0.7,
)
print(result.text)
print(result.prompt_tokens, result.completion_tokens, result.finish_reason)

I can help you brainstorm ideas and develop strategies for growth and development.

21 15 stop


## `LocalGemma.complete(..., stream=True)`

In [27]:
for piece in llm.complete("List three colors:\n", max_tokens=6000, stream=True):
    print(piece, end="", flush=True)
print()

*   **Red**
*   **Blue**
*   **Green**

This is a simple example of a color scheme.

**Explanation:**

The color scheme is a visual representation of the color distribution within a system or environment. It's a visual representation that allows us to understand the relationships between different colors and their properties. The colors are arranged in a way that makes sense for the system's purpose.



## `LocalGemma.chat(..., stream=True)`

In [10]:
for piece in llm.chat(
    [{"role": "user", "content": "Say hi in five words or fewer."}],
    max_tokens=32,
    stream=True,
):
    print(piece, end="", flush=True)
print()

Hi

!



## `LocalLLMClient` (preferred for other files)
Thin wrapper over `LocalGemma`. Pass a shared engine so the model is not loaded twice.

In [28]:
client = LocalLLMClient(in_process=True, engine=llm)
client.model, client.in_process

('gemma3:270m', True)

## `LocalLLMClient.chat`

In [29]:
r = client.chat(
    [{"role": "user", "content": "hi"}],
    max_tokens=64,
)
print(r.text)
print(r.prompt_tokens, r.completion_tokens)

Hi! How are you?

10 7


## `LocalLLMClient.complete`

In [30]:
r = client.complete("The capital of France is", max_tokens=8, temperature=0.2)
print(repr(r.text))
print(r.finish_reason)

' Paris.\nThe capital of France is'
length


## `LocalLLMClient.chat(..., stream=True)`

In [71]:
parts = []
for piece in client.chat(
    [{"role": "user", "content": "Name two programming languages."}],
    max_tokens=48,
    stream=True,
    temperature=0.1,
):
    parts.append(piece)
    print(piece, end="", flush=True)
print()
print("joined:", "".join(parts))

Two programming languages are:

*   **Python**
*   **Java**
joined: Two programming languages are:

*   **Python**
*   **Java**


## Drop-in pattern for other modules
Copy this shape into harness / tower code.

In [87]:
def classify_short(text: str, client: LocalLLMClient | None = None) -> str:
    """Example reusable helper other files can call."""
    c = client or LocalLLMClient(in_process=True)
    out = c.chat(
        [
            {"role": "user", "content": f"Answer correctly with exactly: okay or NO for query: {text}"},
        ],
        max_tokens=100,
        temperature=0.1,
    )
    return (out.text or "").strip().lower()

classify_short("ZBadf", client=client)

'okay'

## Optional cleanup
Free the loaded model when done experimenting.

In [13]:
llm.close()
print("closed")

closed
